In [1]:
!pip install nltk

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [5]:
df = pd.read_csv("reviews.csv")

df.head()

,Review,Sentiment
0,"Loved it, worth every penny",positive
1,Worst purchase ever,negative
2,Superb performance,positive
3,Not recommended,negative
4,Very satisfied with this purchase,positive


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Review     200 non-null    object
 1   Sentiment  200 non-null    object
dtypes: object(2)
memory usage: 3.3+ KB


In [7]:
df['Sentiment'].value_counts()

,count
Sentiment,
positive,100
negative,100


In [8]:
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Convert to lowercase
    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stopwords.words('english')
    ]

    return " ".join(words)

In [9]:
df["Cleaned_Review"] = df["Review"].apply(preprocess)

df.head()

,Review,Sentiment,Cleaned_Review
0,"Loved it, worth every penny",positive,loved worth every penny
1,Worst purchase ever,negative,worst purchase ever
2,Superb performance,positive,superb performance
3,Not recommended,negative,recommended
4,Very satisfied with this purchase,positive,satisfied purchase


In [10]:
tfidf = TfidfVectorizer(max_features=1000)

X = tfidf.fit_transform(df["Cleaned_Review"])

y = df["Sentiment"]

print(X.shape)

(200, 39)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (160, 39)
Testing Data: (40, 39)


In [12]:
model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [13]:
y_pred = model.predict(X_test)

print(y_pred[:10])

['negative' 'negative' 'positive' 'positive' 'positive' 'negative'
 'negative' 'positive' 'positive' 'negative']


In [14]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        16
    positive       1.00      1.00      1.00        24

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [16]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[16  0]
 [ 0 24]]


In [17]:
new_review = ["This product is excellent. I really loved it."]

# Preprocess the review
clean_review = preprocess(new_review[0])

# Convert to TF-IDF
vector = tfidf.transform([clean_review])

# Predict
prediction = model.predict(vector)

print("Review:", new_review[0])
print("Predicted Sentiment:", prediction[0])

Review: This product is excellent. I really loved it.
Predicted Sentiment: positive


In [18]:
new_review = ["Worst product I have ever bought. Totally disappointed."]

clean_review = preprocess(new_review[0])
vector = tfidf.transform([clean_review])

prediction = model.predict(vector)

print("Review:", new_review[0])
print("Predicted Sentiment:", prediction[0])

Review: Worst product I have ever bought. Totally disappointed.
Predicted Sentiment: negative


In [19]:
import pickle

pickle.dump(model, open("sentiment_model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf_vectorizer.pkl", "wb"))

print("Model saved successfully!")

Model saved successfully!
